In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
import warnings

warnings.filterwarnings('ignore')

# Load the dataset
file_path = r"C:\Users\Bala murukan\Desktop\Aadhaar_Insights\Aadhaar_Insight\Aadhaar_Dataset.csv"

print(f"Loading dataset from: {file_path}")
df = pd.read_csv(file_path)

print(f"Dataset loaded successfully with shape: {df.shape}")
df.head()

Loading dataset from: C:\Users\Bala murukan\Desktop\Aadhaar_Insights\Aadhaar_Insight\Aadhaar_Dataset.csv
Dataset loaded successfully with shape: (994402, 14)


,date,state,district,pincode,bio_age_5_17,bio_age_17_,demo_age_5_17,demo_age_17_,age_0_5,age_5_17,age_18_greater,year,month,day
0,2025-01-03,Andaman and Nicobar Islands,Andamans,744101,16.0,193.0,0.0,0.0,0.0,0.0,0.0,2025,1,3
1,2025-01-03,Andaman and Nicobar Islands,Nicobar,744301,101.0,48.0,16.0,180.0,0.0,0.0,0.0,2025,1,3
2,2025-01-03,Andaman and Nicobar Islands,Nicobar,744302,15.0,12.0,0.0,0.0,0.0,0.0,0.0,2025,1,3
3,2025-01-03,Andaman and Nicobar Islands,Nicobar,744303,46.0,27.0,0.0,0.0,0.0,0.0,0.0,2025,1,3
4,2025-01-03,Andaman and Nicobar Islands,Nicobar,744304,16.0,14.0,0.0,0.0,0.0,0.0,0.0,2025,1,3


In [ ]:
# Create total_population
df['total_population'] = df['age_0_5'] + df['age_5_17'] + df['age_18_greater']

# Create estimated_voters
df['estimated_voters'] = df['age_18_greater']

# Create dependency_ratio 
df['dependency_ratio'] = np.where(df['age_18_greater'] == 0, np.nan, (df['age_0_5'] + df['age_5_17']) / df['age_18_greater'])

# Create children_ratio
df['children_ratio'] = np.where(df['total_population'] == 0, np.nan, (df['age_0_5'] + df['age_5_17']) / df['total_population'])

# Create adult_ratio
df['adult_ratio'] = np.where(df['total_population'] == 0, np.nan, df['age_18_greater'] / df['total_population'])

# Create growth_indicator
df['growth_indicator'] = df['bio_age_17_'] - df['demo_age_17_']


1. Basic feature engineering completed.


### 2. Advanced Features

In [ ]:
# youth_ratio
df['youth_ratio'] = np.where(df['total_population'] == 0, np.nan, df['age_5_17'] / df['total_population'])

# aging_index
df['aging_index'] = np.where(df['age_0_5'] == 0, np.nan, df['age_18_greater'] / df['age_0_5'])

# population_share
total_pop_sum = df['total_population'].sum()
df['population_share'] = np.where(total_pop_sum == 0, 0, df['total_population'] / total_pop_sum)

# bio_demo_ratio
df['bio_demo_ratio'] = np.where(df['demo_age_17_'] == 0, np.nan, df['bio_age_17_'] / df['demo_age_17_'])




2. Advanced features engineered.


In [ ]:
# Replace infinite values with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill missing values with 0
df.fillna(0, inplace=True)


4. Edge cases handled: Infinite values replaced with NaN, and missing values filled with 0.


In [ ]:
# 1. Encoding: Convert state and district into numeric values
le_state = LabelEncoder()
df['state_encoded'] = le_state.fit_transform(df['state'].astype(str))

le_district = LabelEncoder()
df['district_encoded'] = le_district.fit_transform(df['district'].astype(str))
# 2. Scaling: Apply Min-Max scaling to total_population
scaler_minmax = MinMaxScaler()
df['total_population_scaled'] = scaler_minmax.fit_transform(df[['total_population']])

# 3. Standardization: Apply Z-score normalization to total_population
scaler_std = StandardScaler()
df['total_population_standardized'] = scaler_std.fit_transform(df[['total_population']])

# 4. Binning: Categorize total_population into Low, Medium, High
df['total_population_binned'] = pd.cut(df['total_population'], bins=3, labels=['Low', 'Medium', 'High'])

# 5. Transformation: Apply log transformation on total_population
df['total_population_log'] = np.log1p(df['total_population'])

print("3. Feature engineering techniques applied (Encoding, Scaling, Standardization, Binning, Transformation).")

3. Feature engineering techniques applied (Encoding, Scaling, Standardization, Binning, Transformation).


### 5. Data Analysis - Results

#### Top 10 States by Total Population

In [18]:
top_states = df.groupby('state')['total_population'].sum().nlargest(10)
top_states

state
Uttar Pradesh     554062.0
Bihar             299949.0
Madhya Pradesh    232136.0
Maharashtra       171930.0
Gujarat           166815.0
West Bengal       164568.0
Rajasthan         154860.0
Assam             143815.0
Karnataka          96232.0
Meghalaya          94096.0
Name: total_population, dtype: float64

#### Top 10 Districts by Estimated Voters

In [19]:
top_districts_voters = df.groupby('district')['estimated_voters'].sum().nlargest(10)
top_districts_voters

district
East Khasi Hills      8065.0
Bengaluru Urban       6331.0
West Khasi Hills      4334.0
West Garo Hills       4123.0
West Jaintia Hills    3285.0
Ri Bhoi               2834.0
Sitamarhi             2301.0
Banaskantha           1930.0
Dahod                 1777.0
Bahraich              1655.0
Name: estimated_voters, dtype: float64

#### Total Youth vs Adult Population

In [20]:
total_youth = df['age_5_17'].sum()
total_adults = df['age_18_greater'].sum()
print(f"Total Youth (5-17): {total_youth:,.0f}")
print(f"Total Adults (18+): {total_adults:,.0f}")

Total Youth (5-17): 935,216
Total Adults (18+): 114,666


#### Top 10 Districts with Highest Dependency Ratio

In [21]:
df_dependency_valid = df[df['dependency_ratio'] > 0]
if not df_dependency_valid.empty:
    district_dependency = df_dependency_valid.groupby('district')['dependency_ratio'].mean().nlargest(10)
    district_dependency
else:
    print("Not enough non-zero dependency data available.")

#### Total Bio vs Demo Population (Adults 18+)

In [22]:
total_bio_18 = df['bio_age_17_'].sum()
total_demo_18 = df['demo_age_17_'].sum()
print(f"Total Bio (18+): {total_bio_18:,.0f}")
print(f"Total Demo (18+): {total_demo_18:,.0f}")

Total Bio (18+): 28,397,704
Total Demo (18+): 21,288,116


In [23]:
output_filename = "Final_Processed_Dataset.csv"
df.to_csv(output_filename, index=False)
print(f"\n✅ Process successfully completed! Final dataset saved as '{output_filename}'")


✅ Process successfully completed! Final dataset saved as 'Final_Processed_Dataset.csv'
